# PokeChess Demo

Demonstrates the PokeChess game engine: board layout, piece movement, attacks, damage, Foresight, evolution, and game flow. All sections include an interactive board view — use the slider to step through board states.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os, warnings
sys.path.insert(0, os.path.dirname(os.getcwd()))
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
import urllib.request

from engine import GameState, Piece, PieceType, Team, Item, PIECE_STATS
from engine.state import KING_TYPES, PAWN_TYPES, ForesightEffect
from engine.moves import get_legal_moves, Move, ActionType
from engine.rules import apply_move, is_terminal, hp_winner

print('Engine imported successfully.')


## Sprite Loader

Sprites are fetched from the public PokeAPI CDN and cached in `demo/sprites/`.

In [ ]:
SPRITE_DIR = os.path.join(os.getcwd(), 'sprites')
os.makedirs(SPRITE_DIR, exist_ok=True)

_POKEMON_DEX = {
    PieceType.SQUIRTLE:   7,
    PieceType.CHARMANDER: 4,
    PieceType.BULBASAUR:  1,
    PieceType.MEW:        151,
    PieceType.PIKACHU:    25,
    PieceType.RAICHU:     26,
    PieceType.EEVEE:      133,
    PieceType.VAPOREON:   134,
    PieceType.FLAREON:    136,
    PieceType.LEAFEON:    470,
    PieceType.JOLTEON:    135,
    PieceType.ESPEON:     196,
    PieceType.POKEBALL:   None,
    PieceType.MASTERBALL: None,
}

SPRITE_SIZE = 64
_sprite_cache = {}


def _fetch_sprite(piece_type):
    if piece_type in _sprite_cache:
        return _sprite_cache[piece_type]
    dex = _POKEMON_DEX.get(piece_type)
    if dex is None:
        # Draw a Pokeball / Masterball circle placeholder
        img = np.zeros((SPRITE_SIZE, SPRITE_SIZE, 4), dtype=np.uint8)
        cx = cy = SPRITE_SIZE // 2
        r = SPRITE_SIZE // 2 - 4
        for y in range(SPRITE_SIZE):
            for x in range(SPRITE_SIZE):
                if (x - cx) ** 2 + (y - cy) ** 2 <= r ** 2:
                    top = y < cy
                    if piece_type == PieceType.MASTERBALL:
                        col = [130, 50, 220, 255] if top else [240, 240, 240, 255]
                    else:
                        col = [220, 50, 50, 255] if top else [240, 240, 240, 255]
                    img[y, x] = col
        _sprite_cache[piece_type] = img
        return img
    path = os.path.join(SPRITE_DIR, f'{dex}.png')
    if not os.path.exists(path):
        url = f'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{dex}.png'
        urllib.request.urlretrieve(url, path)
    arr = np.array(Image.open(path).convert('RGBA').resize((SPRITE_SIZE, SPRITE_SIZE), Image.NEAREST))
    _sprite_cache[piece_type] = arr
    return arr


for pt in PieceType:
    _fetch_sprite(pt)
print(f'Sprites ready ({SPRITE_SIZE}x{SPRITE_SIZE}px each).')


## Board Renderer

In [ ]:
CELL = SPRITE_SIZE + 8
BOARD_PX = CELL * 8
_LIGHT = np.array([240, 217, 181, 255], dtype=np.uint8)
_DARK  = np.array([181, 136,  99, 255], dtype=np.uint8)
_RED_BORDER  = (220,  50,  50)
_BLUE_BORDER = ( 50, 100, 220)
_HIGHLIGHT   = (255, 230,  50)


def _count_duplicates(state):
    """Return (index_map, totals_map) for duplicate-numbering."""
    totals, indices = {}, {}
    for r in range(8):
        for c in range(8):
            p = state.board[r][c]
            if p is None:
                continue
            key = (p.team, p.piece_type)
            totals[key] = totals.get(key, 0) + 1
            indices[(r, c)] = totals[key]
    return indices, totals


def render_board(state, highlight=None, title=None, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6, 6))

    canvas = np.zeros((BOARD_PX, BOARD_PX, 4), dtype=np.uint8)

    # 1. Checkerboard
    for r in range(8):
        for c in range(8):
            y0, x0 = (7 - r) * CELL, c * CELL
            canvas[y0:y0+CELL, x0:x0+CELL] = _LIGHT if (r + c) % 2 == 0 else _DARK

    # 2. Highlights
    for (hr, hc) in (highlight or []):
        y0, x0 = (7 - hr) * CELL, hc * CELL
        base = canvas[y0:y0+CELL, x0:x0+CELL].astype(float)
        overlay = np.array(_HIGHLIGHT, dtype=float)
        canvas[y0:y0+CELL, x0:x0+CELL, :3] = np.clip(
            base[:, :, :3] * 0.55 + overlay * 0.45, 0, 255).astype(np.uint8)

    # 3. Sprites
    for r in range(8):
        for c in range(8):
            p = state.board[r][c]
            if p is None:
                continue
            sp = _fetch_sprite(p.piece_type).astype(float)
            y0, x0 = (7 - r) * CELL + 4, c * CELL + 4
            bg = canvas[y0:y0+SPRITE_SIZE, x0:x0+SPRITE_SIZE].astype(float)
            a = sp[:, :, 3:4] / 255.0
            canvas[y0:y0+SPRITE_SIZE, x0:x0+SPRITE_SIZE, :3] = np.clip(
                sp[:, :, :3] * a + bg[:, :, :3] * (1 - a), 0, 255).astype(np.uint8)
            canvas[y0:y0+SPRITE_SIZE, x0:x0+SPRITE_SIZE, 3] = 255

    ax.imshow(canvas)

    # 4. Coloured borders + duplicate numbers
    indices, totals = _count_duplicates(state)
    for r in range(8):
        for c in range(8):
            p = state.board[r][c]
            if p is None:
                continue
            ec = [v/255 for v in (_RED_BORDER if p.team == Team.RED else _BLUE_BORDER)]
            ax.add_patch(mpatches.FancyBboxPatch(
                (c * CELL + 1, (7 - r) * CELL + 1), CELL - 2, CELL - 2,
                boxstyle='square,pad=0', lw=2.5, edgecolor=ec, facecolor='none'))
            key = (p.team, p.piece_type)
            if totals.get(key, 1) > 1:
                ax.text(c * CELL + CELL - 4, (7 - r) * CELL + 12, str(indices[(r, c)]),
                        fontsize=7, fontweight='bold', color=ec, ha='right', va='top')

    # 5. Axis labels
    ax.set_xticks([c * CELL + CELL // 2 for c in range(8)])
    ax.set_yticks([(7 - r) * CELL + CELL // 2 for r in range(8)])
    ax.set_xticklabels(range(8), fontsize=8)
    ax.set_yticklabels(range(8), fontsize=8)
    ax.tick_params(length=0)
    ax.set_xlim(0, BOARD_PX)
    ax.set_ylim(BOARD_PX, 0)
    ax.set_title(title or f'Turn {state.turn_number}  |  Active: {state.active_player.name}', fontsize=10)

    if standalone:
        plt.tight_layout()
        plt.show()


print('Renderer ready.')


## Interactive Viewer

In [ ]:
def show_states(states_labels, highlight_fn=None, figsize=(6, 6)):
    n = len(states_labels)
    if n == 0:
        return
    if n == 1:
        state, label = states_labels[0]
        render_board(state, title=label, figsize=figsize)
        return
    out = widgets.Output()
    slider = widgets.IntSlider(value=0, min=0, max=n - 1, step=1,
                                description='Step:', continuous_update=False,
                                layout=widgets.Layout(width='500px'))
    def _draw(change):
        idx = change['new'] if isinstance(change, dict) else change
        state, label = states_labels[idx]
        hl = highlight_fn(state, idx) if highlight_fn else None
        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=figsize)
            render_board(state, highlight=hl, title=label, ax=ax)
            plt.tight_layout()
            plt.show()
    slider.observe(_draw, names='value')
    display(widgets.VBox([slider, out]))
    _draw(0)


print('Interactive viewer ready.')


## Helpers

In [ ]:
def empty_state(active=Team.RED, turn=1):
    board = [[None] * 8 for _ in range(8)]
    return GameState(board=board, active_player=active, turn_number=turn,
                     pending_foresight={Team.RED: None, Team.BLUE: None})

def place(state, pt, team, row, col):
    p = Piece.create(pt, team, row, col)
    state.board[row][col] = p
    return p

def apply_det(state, row, col, action_type, target_row, target_col, **kw):
    move = Move(piece_row=row, piece_col=col, action_type=action_type,
                target_row=target_row, target_col=target_col, **kw)
    return apply_move(state, move)[0][0]

def moves_from(state, row, col):
    return [m for m in get_legal_moves(state) if m.piece_row == row and m.piece_col == col]

print('Helpers ready.')


---
## Section 1 — Starting Board

RED (Pikachu side) occupies rows 0–1; BLUE (Eevee side) occupies rows 6–7. Numbers in the corner distinguish identical pieces on the same side.

In [ ]:
render_board(GameState.new_game(), title='Starting Position')

---
## Section 2 — Legal Move Inspection

Yellow highlights = legal move targets for the selected piece. Use the slider to view different pieces.

In [ ]:
s = GameState.new_game()
pieces = [
    (1, 3, 'Red Pokeball (1,3): forward, lateral (up to 2), forward-diagonal'),
    (0, 4, 'Red Pikachu (0,4): 1-square in all 8 directions'),
    (0, 3, 'Red Mew (0,3): queen — slides all 8 directions (blocked by Pokeball row)'),
    (0, 0, 'Red Squirtle (0,0): rook — file blocked by Pokeball at (1,0)'),
    (0, 1, 'Red Charmander (0,1): knight — jumps in L-shapes over pieces'),
    (0, 2, 'Red Bulbasaur (0,2): bishop — diagonals blocked by Pokeballs'),
]
sl = []
hl = []
for row, col, lbl in pieces:
    mvs = moves_from(s, row, col)
    sl.append((s, lbl))
    hl.append([(m.target_row, m.target_col) for m in mvs])
show_states(sl, highlight_fn=lambda st, i: hl[i])


---
## Section 3 — Pokeball Movement

Pokeballs differ from chess pawns:
- Move **up to 2 squares forward** (not limited to 1 after leaving home row)
- Move **up to 2 squares laterally** (left or right)
- Move **1 square forward-diagonally**
- **No backward movement** (Masterball adds that)

All these also apply as attack directions if an enemy occupies that square.

In [ ]:
pb_s = empty_state()
place(pb_s, PieceType.PIKACHU, Team.RED,  0, 4)
place(pb_s, PieceType.EEVEE,   Team.BLUE, 7, 4)
place(pb_s, PieceType.POKEBALL, Team.RED,  3, 3)

mvs = moves_from(pb_s, 3, 3)
move_tgts = [(m.target_row, m.target_col) for m in mvs if m.action_type == ActionType.MOVE]

# Walk through several moves to show the Pokeball can still move 2 squares
# in any of the valid directions from mid-board
s0 = pb_s
s1 = apply_det(s0, 3, 3, ActionType.MOVE, 4, 3)  # 1 forward
s1b = apply_det(s1, 7, 4, ActionType.MOVE, 6, 4)  # blue responds
s2 = apply_det(s1b, 4, 3, ActionType.MOVE, 5, 3)  # 2nd forward (not home row — still works)
s2b = apply_det(s2, 6, 4, ActionType.MOVE, 5, 4)  # blue responds
s3 = apply_det(s2b, 5, 3, ActionType.MOVE, 5, 5)  # 2 lateral right
s3b = apply_det(s3, 5, 4, ActionType.MOVE, 4, 4)  # blue responds
s4 = apply_det(s3b, 5, 5, ActionType.MOVE, 5, 3)  # 2 lateral left
s4b = apply_det(s4, 4, 4, ActionType.MOVE, 4, 5)  # blue responds
s5 = apply_det(s4b, 5, 3, ActionType.MOVE, 6, 4)  # forward-right diagonal

show_states([
    (s0,  'Start: Pokeball at (3,3) — yellow = all legal moves'),
    (s1b, 'After 1 square forward → (4,3)'),
    (s2b, 'After 2nd forward (mid-board) → (5,3)  — Pokeballs can always move 2 fwd'),
    (s3b, 'After 2 lateral right → (5,5)'),
    (s4b, 'After 2 lateral left → (5,3)'),
    (s5,  'After forward-right diagonal → (6,4)'),
], highlight_fn=lambda st, i: move_tgts if i == 0 else [])


---
## Section 4 — Squirtle (Rook) Movement

Squirtle slides any number of squares along ranks or files. Friendly pieces block it; the first enemy on a ray can be attacked but the ray goes no further.

In [ ]:
sq_s = empty_state()
place(sq_s, PieceType.PIKACHU,    Team.RED,  0, 4)
place(sq_s, PieceType.EEVEE,      Team.BLUE, 7, 4)
place(sq_s, PieceType.SQUIRTLE,   Team.RED,  0, 0)   # the piece we're moving
place(sq_s, PieceType.SQUIRTLE,   Team.RED,  0, 3)   # friendly blocker on the rank
place(sq_s, PieceType.CHARMANDER, Team.BLUE, 4, 0)   # enemy on the file

mvs_sq = moves_from(sq_s, 0, 0)
sq_tgts = [(m.target_row, m.target_col) for m in mvs_sq]

# Move Squirtle up the file; it hits the Blue Charmander at (4,0)
sq2 = apply_det(sq_s, 0, 0, ActionType.MOVE, 3, 0)
sq2b = apply_det(sq2, 7, 4, ActionType.MOVE, 6, 4)

# Now attack the Charmander
sq3 = apply_det(sq2b, 3, 0, ActionType.ATTACK, 4, 0)
sq3b = apply_det(sq3, 6, 4, ActionType.MOVE, 5, 4)

# Slide along a rank
sq4 = apply_det(sq3b, 4, 0, ActionType.MOVE, 4, 5)

show_states([
    (sq_s,  'Squirtle (1) at (0,0) — highlights show legal squares\nFriendly at (0,3) blocks rank; enemy Charmander at (4,0) is attack target'),
    (sq2b,  'Squirtle slides to (3,0) — Charmander at (4,0) blocks further progress'),
    (sq3b,  'Squirtle attacks Charmander at (4,0) — 200 dmg (Water 2x Fire) → KO'),
    (sq4,   'Squirtle slides along rank 4 to (4,5)'),
], highlight_fn=lambda st, i: sq_tgts if i == 0 else [])


---
## Section 5 — Combat: Damage and KO

Base damage for starters is **100**. Type multipliers: Water > Fire > Grass > Water (2×/0.5×). Equal or unrelated types = 1×.

In [ ]:
from engine.rules import _calc_damage

def dmg(at, dt, slot=None):
    a = Piece.create(at, Team.RED,  0, 0); a.held_item = Item.NONE
    d = Piece.create(dt, Team.BLUE, 0, 1); d.held_item = Item.NONE
    return _calc_damage(a, d, slot)

print('Damage table (base 100, no held item):')
rows = [
    ('Squirtle', 'Pikachu',    PieceType.SQUIRTLE,   PieceType.PIKACHU,    '1.0x neutral'),
    ('Squirtle', 'Charmander', PieceType.SQUIRTLE,   PieceType.CHARMANDER, '2.0x super-effective -> KO'),
    ('Squirtle', 'Bulbasaur',  PieceType.SQUIRTLE,   PieceType.BULBASAUR,  '0.5x not-very-effective'),
    ('Charmander','Bulbasaur', PieceType.CHARMANDER, PieceType.BULBASAUR,  '2.0x super-effective -> KO'),
    ('Bulbasaur', 'Squirtle',  PieceType.BULBASAUR,  PieceType.SQUIRTLE,   '2.0x super-effective -> KO'),
    ('Mew slot0', 'Pikachu',   PieceType.MEW,        PieceType.PIKACHU,    '1.0x slot 0 = 40 base'),
    ('Mew slot3', 'Squirtle',  PieceType.MEW,        PieceType.SQUIRTLE,   '1.0x slot 3 = 200 base -> KO'),
]
for a_name, d_name, at, dt, note in rows:
    slot = {'Mew slot0': 0, 'Mew slot3': 3}.get(a_name)
    print(f'  {a_name:<12} vs {d_name:<12}: {dmg(at, dt, slot):>4} dmg  ({note})')


In [ ]:
# Multi-step combat scenario
cs = empty_state()
place(cs, PieceType.PIKACHU,    Team.RED,  0, 4)
place(cs, PieceType.EEVEE,      Team.BLUE, 7, 4)
r_sq  = place(cs, PieceType.SQUIRTLE,   Team.RED,  3, 2); r_sq.held_item  = Item.NONE
b_ch  = place(cs, PieceType.CHARMANDER, Team.BLUE, 3, 3); b_ch.held_item  = Item.NONE
b_bu  = place(cs, PieceType.BULBASAUR,  Team.BLUE, 4, 2); b_bu.held_item  = Item.NONE
r_mew = place(cs, PieceType.MEW,        Team.RED,  2, 5); r_mew.held_item = Item.NONE

# Turn 1: Squirtle attacks Charmander (2x -> KO, 200 dmg)
cs2 = apply_det(cs, 3, 2, ActionType.ATTACK, 3, 3)
# Turn 2: Bulbasaur attacks Squirtle (2x -> KO, 200 dmg)
cs3 = apply_det(cs2, 4, 2, ActionType.ATTACK, 3, 3)
# Turn 3: Mew slot-3 attacks Bulbasaur (neutral, 200 dmg -> KO)
cs4 = apply_det(cs3, 2, 5, ActionType.ATTACK, 4, 3, move_slot=3)

show_states([
    (cs,  'Setup: Squirtle(R), Charmander(B), Bulbasaur(B), Mew(R)'),
    (cs2, 'Squirtle attacks Charmander — 200 dmg (Water 2x Fire) → KO'),
    (cs3, 'Bulbasaur attacks Squirtle — 200 dmg (Grass 2x Water) → KO'),
    (cs4, 'Mew slot-3 attacks Bulbasaur — 200 dmg (Psychic 1x Grass) → KO'),
])


---
## Section 6 — Items (Evolution Only)

Held items gate evolution but have **no effect on damage**. Pokeballs and Masterballs cannot hold items and therefore cannot trade.

In [ ]:
for item in [Item.NONE, Item.WATERSTONE, Item.FIRESTONE]:
    a = Piece.create(PieceType.SQUIRTLE, Team.RED, 0, 0); a.held_item = item
    d = Piece.create(PieceType.CHARMANDER, Team.BLUE, 0, 1)
    print(f'  Squirtle ({item.name:<12}) -> Charmander: {_calc_damage(a, d)} dmg  (identical regardless of item)')

# Confirm Pokeball has no trade moves
no_trade = empty_state()
place(no_trade, PieceType.POKEBALL, Team.RED, 3, 3)
place(no_trade, PieceType.SQUIRTLE, Team.RED, 3, 4)
pb_mvs = moves_from(no_trade, 3, 3)
trade_mvs = [m for m in pb_mvs if m.action_type == ActionType.TRADE]
print(f'\nPokeball trade moves: {len(trade_mvs)}  (Pokeballs cannot hold items -> no trades)')


---
## Section 7 — Stochastic Pokeball Capture

Two outcomes at p=0.5 each. Pikachu/Raichu are immune (deterministic fail). Masterball always captures.

In [ ]:
pb_d = empty_state()
place(pb_d, PieceType.PIKACHU,  Team.RED,  0, 4)
place(pb_d, PieceType.EEVEE,    Team.BLUE, 7, 4)
place(pb_d, PieceType.POKEBALL, Team.RED,  3, 3)
place(pb_d, PieceType.SQUIRTLE, Team.BLUE, 3, 4)

atk_move = Move(piece_row=3, piece_col=3, action_type=ActionType.ATTACK,
                target_row=3, target_col=4)
outcomes = apply_move(pb_d, atk_move)

sl = [(pb_d, 'Before: Red Pokeball attacks Blue Squirtle')]
for i, (ns, prob) in enumerate(outcomes):
    sq_a = ns.board[3][4]
    result = 'Capture succeeded' if sq_a and sq_a.team == Team.RED else 'Capture failed'
    print(f'Outcome {i+1}: p={prob} -> {result}')
    sl.append((ns, f'Outcome {i+1} (p={prob}): {result}'))

# Pikachu immunity
pika_d = empty_state()
place(pika_d, PieceType.PIKACHU, Team.RED,  0, 4)
place(pika_d, PieceType.EEVEE,   Team.BLUE, 7, 4)
place(pika_d, PieceType.POKEBALL, Team.RED,  3, 3)
place(pika_d, PieceType.PIKACHU,  Team.BLUE, 3, 4)
pika_atk = Move(piece_row=3, piece_col=3, action_type=ActionType.ATTACK, target_row=3, target_col=4)
pika_outcomes = apply_move(pika_d, pika_atk)
print(f'\nPokeball vs Pikachu: {len(pika_outcomes)} outcome (immune)')
sl.append((pika_d, f'Pokeball vs Pikachu: {len(pika_outcomes)} outcome — immune (deterministic fail)'))

show_states(sl)


---
## Section 8 — Foresight (Mew)

Foresight schedules delayed damage that resolves at the **start of the caster's next turn** (2 half-moves later). Cannot be cast on consecutive turns.

In [ ]:
fs = empty_state(turn=1)
place(fs, PieceType.PIKACHU,  Team.RED,  0, 4)
place(fs, PieceType.EEVEE,    Team.BLUE, 7, 4)
place(fs, PieceType.MEW,      Team.RED,  3, 3)
place(fs, PieceType.SQUIRTLE, Team.BLUE, 5, 5)

sq_hp = fs.board[5][5].current_hp

# Turn 1: Mew casts Foresight on (5,5)
fs1 = apply_det(fs, 3, 3, ActionType.FORESIGHT, 5, 5)
fx = fs1.pending_foresight[Team.RED]

# Turn 2: Blue moves king
fs2 = apply_det(fs1, 7, 4, ActionType.MOVE, 6, 4)

# Turn 3: Red moves king — Foresight resolves at start of turn before this move
fs3 = apply_det(fs2, 0, 4, ActionType.MOVE, 0, 3)
sq_a = fs3.board[5][5]
result = 'KO' if sq_a is None else f'HP {sq_a.current_hp}/{sq_hp}'

print(f'Foresight: damage={fx.damage}, resolves_on_turn={fx.resolves_on_turn}')
print(f'Squirtle result after resolution: {result}')

show_states([
    (fs,  'Turn 1: Mew and Blue Squirtle at (5,5)'),
    (fs1, f'Turn 1 done: Foresight cast — damage={fx.damage}, resolves turn {fx.resolves_on_turn}'),
    (fs2, 'Turn 2: Blue moves — Foresight still pending'),
    (fs3, f'Turn 3: Foresight resolves — Squirtle {result}'),
])


---
## Section 9 — King Evolution

Evolution costs a turn. HP increases by the **HP delta** (evolved max HP − base max HP). An injured king also gains the full delta, not just capped at the new max.

In [ ]:
# Full-HP and injured Pikachu
ev1 = empty_state()
pika_full    = place(ev1, PieceType.PIKACHU, Team.RED, 2, 2)  # 200 HP
pika_injured = place(ev1, PieceType.PIKACHU, Team.RED, 5, 5)
pika_injured.current_hp = 80  # injured

ev2 = apply_det(ev1, 2, 2, ActionType.EVOLVE, 2, 2)  # evolve full-HP Pikachu
# skip to Blue turn, then evolve injured
# (both are RED so only one evolves per turn — use two separate demos)
r_full = ev2.board[2][2]
print(f'Pikachu (200 HP full) -> Raichu: {r_full.current_hp}/250  (+50 delta)')

ev_inj = empty_state()
inj = place(ev_inj, PieceType.PIKACHU, Team.RED, 5, 5); inj.current_hp = 80
ev_inj2 = apply_det(ev_inj, 5, 5, ActionType.EVOLVE, 5, 5)
r_inj = ev_inj2.board[5][5]
print(f'Pikachu (80 HP injured) -> Raichu: {r_inj.current_hp}/250  (80 + 50 delta)')

# Eevee evolution with stone
ev3 = empty_state(active=Team.BLUE)
eevee = place(ev3, PieceType.EEVEE, Team.BLUE, 4, 4)
eevee.held_item = Item.WATERSTONE
ev4 = apply_det(ev3, 4, 4, ActionType.EVOLVE, 4, 4, move_slot=0)
vap = ev4.board[4][4]
print(f'Eevee (120 HP) + WATERSTONE -> Vaporeon: {vap.current_hp}/220  (+100 delta, stone consumed, item={vap.held_item.name})')

show_states([
    (ev1,  'Before: two Pikachus (full + injured at 80 HP)'),
    (ev2,  f'Full Pikachu -> Raichu: {r_full.current_hp}/250'),
    (ev_inj2, f'Injured Pikachu -> Raichu: {r_inj.current_hp}/250 (80+50)'),
    (ev3,  'Before Eevee evolution (WATERSTONE)'),
    (ev4,  f'Eevee -> Vaporeon: {vap.current_hp}/220 (+100, stone gone)'),
])


---
## Section 10 — Win Condition and HP Tiebreaker

Game ends when a king is eliminated. Tiebreaker: total HP across all pieces, where **Pokeball = 50** and **Masterball = 200** (fixed values, not their actual HP which is 0).

In [ ]:
# Win condition
w = empty_state()
place(w, PieceType.PIKACHU, Team.RED,  0, 4)
place(w, PieceType.EEVEE,   Team.BLUE, 7, 4)
done, winner = is_terminal(w)
print(f'Both kings alive: terminal={done}')

w2 = w.copy(); w2.board[7][4] = None
done2, winner2 = is_terminal(w2)
print(f'Blue king removed: terminal={done2}, winner={winner2.name}')

# HP tiebreaker with Pokeball/Masterball fixed values
hp_s = empty_state()
r_sq = place(hp_s, PieceType.SQUIRTLE,  Team.RED,  1, 0); r_sq.current_hp = 80
place(hp_s, PieceType.POKEBALL,   Team.RED,  1, 1)    # fixed 50
place(hp_s, PieceType.MASTERBALL, Team.BLUE, 6, 0)    # fixed 200
b_sq = place(hp_s, PieceType.SQUIRTLE, Team.BLUE, 6, 1); b_sq.current_hp = 60

print(f'\nHP tiebreaker:')
print(f'  RED:  Squirtle(80) + Pokeball(50 fixed) = 130')
print(f'  BLUE: Masterball(200 fixed) + Squirtle(60) = 260')
print(f'  Winner: {hp_winner(hp_s).name}')

show_states([
    (w,    'Both kings alive — game ongoing'),
    (w2,   'Blue king eliminated — RED wins'),
    (hp_s, 'HP tiebreaker scenario: RED=130, BLUE=260 -> BLUE wins'),
])


---
## Section 11 — Mini Game: 20 Random Moves

Random move selection from the starting position. Use the slider to replay every board state.

In [ ]:
import random
random.seed(42)

game = GameState.new_game()
game_states = [(game, 'Turn 1 — Starting position')]

for _ in range(20):
    done, winner = is_terminal(game)
    if done:
        break
    moves = get_legal_moves(game)
    if not moves:
        break
    move = random.choice(moves)
    outcomes = apply_move(game, move)
    game = random.choices([s for s, _ in outcomes], weights=[p for _, p in outcomes])[0]
    team_played = 'RED' if game.active_player == Team.BLUE else 'BLUE'
    label = (f'Turn {game.turn_number}  |  {team_played} played '
             f'{move.action_type.name} ({move.piece_row},{move.piece_col})->'
             f'({move.target_row},{move.target_col})')
    game_states.append((game, label))

done, winner = is_terminal(game)
if done:
    last_s, last_l = game_states[-1]
    game_states[-1] = (last_s, last_l + f'\n★ GAME OVER — {winner.name if winner else "DRAW"} wins')

print(f'{len(game_states)} states. Use the slider to replay.')
show_states(game_states)
